In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abirsaafi/cardioagent-cardiovascular-instructions/__huggingface_repos__.json
/kaggle/input/datasets/abirsaafi/cardioagent-cardiovascular-instructions/cardioagent_dataset/test.jsonl
/kaggle/input/datasets/abirsaafi/cardioagent-cardiovascular-instructions/cardioagent_dataset/train.jsonl
/kaggle/input/datasets/abirsaafi/cardioagent-cardiovascular-instructions/cardioagent_dataset/validation.jsonl


In [2]:
# CELL 1 — Install dependencies (run once)
import sys
import subprocess

packages = [
    "unsloth[kaggle-new]",
    "mlflow",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U", *packages
])

print("Installation terminée.")
print("Si Kaggle demande un redémarrage, redémarre UNE seule fois, puis commence à la cellule 2.")

Installation terminée.
Si Kaggle demande un redémarrage, redémarre UNE seule fois, puis commence à la cellule 2.


In [3]:
# CELL 2 — Imports, environment and configuration
# Import Unsloth BEFORE Transformers, TRL and PEFT.
import os
import sys
import json
import gc
import glob
import math
import random
import importlib.metadata as metadata

import unsloth
import torch
import numpy as np
import mlflow

from unsloth import FastLanguageModel
from datasets import Dataset
from transformers import AutoTokenizer, TrainerCallback
from trl import SFTConfig, SFTTrainer

SEED = 3407
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 2048

# Conservative QLoRA settings after the earlier T4 backward-pass OOM.
LORA_R = 8
LORA_ALPHA = 16
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

EPOCHS = 3
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
LEARNING_RATE = 2e-4

OUTPUT_DIR = "/kaggle/working/cardioagent_qlora"
ADAPTER_DIR = "/kaggle/working/cardioagent_qlora_adapter"
MLFLOW_DB = "/kaggle/working/mlflow.db"
MLFLOW_EXPERIMENT = "CardioAgent-QLoRA"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Unsloth:", metadata.version("unsloth"))
print("Transformers:", metadata.version("transformers"))
print("TRL:", metadata.version("trl"))
print("PEFT:", metadata.version("peft"))
print("MLflow:", mlflow.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("GPU non détecté. Active un GPU dans Kaggle Settings → Accelerator.")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | {props.total_memory / 1024**3:.2f} GB VRAM")

print("\nConfiguration prête.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers does not load with torch 2.11.0+cu130, using PyTorch attention (UNSLOTH_ENABLE_LOGGING=1 for the fix).
🦥 Unsloth Zoo will now patch everything to make training faster!
Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA available: True
Unsloth: 2026.11.12
Transformers: 5.17.0
TRL: 1.15.0
PEFT: 0.21.2
MLflow: 3.18.0
GPU 0: Tesla T4 | 14.56 GB VRAM
GPU 1: Tesla T4 | 14.56 GB VRAM

Configuration prête.


In [5]:
# CELL 3 — Locate and load the prepared JSONL dataset
# Add the Kaggle Dataset as an Input before running this cell.

jsonl_matches = glob.glob(
    "/kaggle/input/**/train.jsonl",
    recursive=True
)

if not jsonl_matches:
    raise FileNotFoundError(
        "train.jsonl introuvable. Dans Kaggle, fais Add Input → Your Datasets "
        "et ajoute cardioagent-cardiovascular-instructions."
    )

# Select the folder containing all three expected splits.
DATASET_DIR = None
for train_path in jsonl_matches:
    candidate_dir = os.path.dirname(train_path)
    expected = ["train.jsonl", "validation.jsonl", "test.jsonl"]
    if all(os.path.isfile(os.path.join(candidate_dir, name)) for name in expected):
        DATASET_DIR = candidate_dir
        break

if DATASET_DIR is None:
    raise FileNotFoundError(
        "J'ai trouvé train.jsonl, mais pas les trois fichiers "
        "train/validation/test dans le même dossier."
    )

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"JSON invalide dans {path}, ligne {line_number}: {exc}")
    return rows

train_raw = load_jsonl(os.path.join(DATASET_DIR, "train.jsonl"))
validation_raw = load_jsonl(os.path.join(DATASET_DIR, "validation.jsonl"))
test_raw = load_jsonl(os.path.join(DATASET_DIR, "test.jsonl"))

for split_name, rows in [
    ("train", train_raw),
    ("validation", validation_raw),
    ("test", test_raw),
]:
    if not rows:
        raise ValueError(f"Le split {split_name} est vide.")
    for i, row in enumerate(rows):
        if "messages" not in row or not isinstance(row["messages"], list):
            raise ValueError(f"Structure messages manquante: {split_name}[{i}]")
        roles = [m.get("role") for m in row["messages"]]
        if not {"user", "assistant"}.issubset(set(roles)):
            raise ValueError(f"Conversation incomplète: {split_name}[{i}], roles={roles}")

print("Dataset folder:", DATASET_DIR)
print("Train:", len(train_raw))
print("Validation:", len(validation_raw))
print("Test:", len(test_raw))
print("Total:", len(train_raw) + len(validation_raw) + len(test_raw))
print("Sample roles:", [m["role"] for m in train_raw[0]["messages"]])

Dataset folder: /kaggle/input/datasets/abirsaafi/cardioagent-cardiovascular-instructions/cardioagent_dataset
Train: 254
Validation: 55
Test: 55
Total: 364
Sample roles: ['system', 'user', 'assistant']


In [6]:
# CELL 4 — Load tokenizer, measure lengths and prepare splits
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if tokenizer.chat_template is None:
    raise ValueError("Le tokenizer ne contient pas de chat template.")

def conversation_length(record):
    return len(tokenizer.apply_chat_template(
        record["messages"],
        tokenize=True,
        add_generation_prompt=False,
    ))

lengths = {
    "train": [conversation_length(x) for x in train_raw],
    "validation": [conversation_length(x) for x in validation_raw],
    "test": [conversation_length(x) for x in test_raw],
}

def keep_complete_examples(rows, row_lengths, split_name):
    kept = [row for row, length in zip(rows, row_lengths) if length <= MAX_SEQ_LENGTH]
    removed = len(rows) - len(kept)
    print(f"{split_name}: {len(rows)} originaux | {len(kept)} conservés | {removed} exclus (> {MAX_SEQ_LENGTH} tokens)")
    if not kept:
        raise ValueError(f"Aucun exemple {split_name} ne respecte MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}.")
    return kept

train_data = keep_complete_examples(train_raw, lengths["train"], "Train")
validation_data = keep_complete_examples(validation_raw, lengths["validation"], "Validation")
test_data = keep_complete_examples(test_raw, lengths["test"], "Test")

# The source JSONL files remain unchanged. Only the in-memory training/evaluation
# datasets are filtered to avoid cutting off the end of long assistant answers.
train_dataset = Dataset.from_list([{"messages": x["messages"]} for x in train_data])
eval_dataset = Dataset.from_list([{"messages": x["messages"]} for x in validation_data])
test_dataset = Dataset.from_list([{"messages": x["messages"]} for x in test_data])

print("\nMaximum kept train length:", max(conversation_length(x) for x in train_data))
print("Maximum kept validation length:", max(conversation_length(x) for x in validation_data))
print("Maximum kept test length:", max(conversation_length(x) for x in test_data))
print("Chat template:", "OK")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Train: 254 originaux | 244 conservés | 10 exclus (> 2048 tokens)
Validation: 55 originaux | 54 conservés | 1 exclus (> 2048 tokens)
Test: 55 originaux | 54 conservés | 1 exclus (> 2048 tokens)

Maximum kept train length: 1952
Maximum kept validation length: 1511
Maximum kept test length: 1365
Chat template: OK


In [7]:
# CELL 5 — Load Qwen 2.5 7B in 4-bit and attach LoRA
# This is the main VRAM-consuming step.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    device_map={"": 0},
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

print("\nModel:", MODEL_NAME)
print("4-bit quantization: enabled")
print("Maximum sequence length:", MAX_SEQ_LENGTH)
print("LoRA rank:", LORA_R)
print("LoRA targets:", LORA_TARGET_MODULES)
print("GPU allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
print("GPU reserved:", round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GB")

==((====))==  Unsloth 2026.11.12: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.11.12 patched 28 layers with 28 QKV layers, 0 O layers and 0 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
trainable params: 2,523,136 || all params: 7,618,139,648 || trainable%: 0.0331

Model: Qwen/Qwen2.5-7B-Instruct
4-bit quantization: enabled
Maximum sequence length: 2048
LoRA rank: 8
LoRA targets: ['q_proj', 'v_proj']
GPU allocated: 6.67 GB
GPU reserved: 6.69 GB


In [8]:
# CELL 6 — Configure MLflow and the SFT trainer
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
mlflow.set_experiment(MLFLOW_EXPERIMENT)

class MLflowMetricsCallback(TrainerCallback):
    # Trainer logs are mirrored into the active MLflow run.
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or mlflow.active_run() is None:
            return
        step = int(state.global_step)
        for key in ("loss", "eval_loss", "learning_rate", "grad_norm", "epoch"):
            value = logs.get(key)
            if isinstance(value, (int, float)) and math.isfinite(value):
                mlflow.log_metric(key, float(value), step=step)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=5,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    callbacks=[MLflowMetricsCallback()],
)

print("Trainer:", type(trainer).__name__)
print("Train examples:", len(train_dataset))
print("Validation examples:", len(eval_dataset))
print("Effective batch size:", TRAIN_BATCH_SIZE * GRAD_ACCUMULATION)
print("Epochs:", EPOCHS)
print("MLflow tracking URI:", mlflow.get_tracking_uri())

2026/11/26 20:01:39 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/26 20:01:39 INFO mlflow.store.db.utils: Updating database tables
2026/11/26 20:01:41 INFO mlflow.tracking.fluent: Experiment with name 'CardioAgent-QLoRA' does not exist. Creating a new experiment.


Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


Unsloth: Tokenizing ["messages"] (num_proc=2):   0%|          | 0/244 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["messages"] (num_proc=2):   0%|          | 0/54 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer: UnslothSFTTrainer
Train examples: 244
Validation examples: 54
Effective batch size: 8
Epochs: 3
MLflow tracking URI: sqlite:////kaggle/working/mlflow.db


In [9]:
# CELL 7 — Run QLoRA fine-tuning and log the experiment
# If CUDA OOM occurs here, stop rather than repeatedly retrying in the same session.
with mlflow.start_run(run_name="qwen25-7b-qlora-cardioagent") as run:
    mlflow.log_params({
        "project": "CardioAgent-Ops",
        "base_model": MODEL_NAME,
        "method": "QLoRA 4-bit",
        "max_seq_length": MAX_SEQ_LENGTH,
        "lora_rank": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_targets": ",".join(LORA_TARGET_MODULES),
        "epochs": EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUMULATION,
        "learning_rate": LEARNING_RATE,
        "train_examples": len(train_dataset),
        "validation_examples": len(eval_dataset),
        "test_examples": len(test_dataset),
    })

    train_result = trainer.train()
    trainer.log_metrics("train", train_result.metrics)
    trainer.save_metrics("train", train_result.metrics)
    trainer.save_state()

    eval_metrics = trainer.evaluate()
    trainer.log_metrics("eval", eval_metrics)
    trainer.save_metrics("eval", eval_metrics)

    print("Training completed.")
    MLFLOW_RUN_ID = run.info.run_id
    print("Run ID:", MLFLOW_RUN_ID)
    print("Train metrics:", train_result.metrics)
    print("Validation metrics:", eval_metrics)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 244 | Num Epochs = 3 | Total steps = 93
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 2,523,136 of 7,618,139,648 (0.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,1.105481,1.082598
2,1.003191,0.968197
3,0.941171,0.949397


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/cardioagent_qlora/checkpoint-31/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/cardioagent_qlora/checkpoint-62/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/cardioagent_qlora/checkpoint-93/tokenizer_config.json.


***** train metrics *****
  epoch                    =        3.0
  total_flos               = 13353074GF
  train_loss               =      1.124
  train_runtime            = 0:20:24.16
  train_samples_per_second =      0.598
  train_steps_per_second   =      0.076


Training Loss,Validation Loss,Epoch
0.941171,0.949397,3


***** eval metrics *****
  eval_loss = 0.9494
Training completed.
Run ID: c6cda821a7754d08bf9d604961f066e2
Train metrics: {'train_runtime': 1224.1674, 'train_samples_per_second': 0.598, 'train_steps_per_second': 0.076, 'total_flos': 1.433775450530304e+16, 'train_loss': 1.123988228459512, 'epoch': 3.0}
Validation metrics: {'eval_loss': 0.949396550655365}


In [10]:
# CELL 8 — Final test evaluation and save the LoRA adapter
with mlflow.start_run(run_id=MLFLOW_RUN_ID):
    test_metrics = trainer.evaluate(
        eval_dataset=test_dataset,
        metric_key_prefix="test",
    )
    trainer.log_metrics("test", test_metrics)
    trainer.save_metrics("test", test_metrics)

    # Save the fine-tuned adapter and tokenizer separately from the base model.
    trainer.save_model(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)

    mlflow.log_params({"adapter_export": "completed"})
    mlflow.log_metrics({
        key: float(value)
        for key, value in test_metrics.items()
        if isinstance(value, (int, float)) and math.isfinite(value)
    })

print("Test metrics:", test_metrics)
print("\nAdapter saved to:", ADAPTER_DIR)
print("Adapter files:", sorted(os.listdir(ADAPTER_DIR)))


Unsloth: Tokenizing ["messages"] (num_proc=2):   0%|          | 0/54 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Epoch
0.941171,0.948079,3


***** test metrics *****
  test_loss = 0.9481


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/cardioagent_qlora_adapter/tokenizer_config.json.


Test metrics: {'test_loss': 0.9480792284011841}

Adapter saved to: /kaggle/working/cardioagent_qlora_adapter
Adapter files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


In [11]:
# CELL 9 — Create portable backup and attach artifacts to the MLflow run
import shutil

ADAPTER_ARCHIVE = "/kaggle/working/cardioagent_qlora_adapter.zip"
shutil.make_archive(
    "/kaggle/working/cardioagent_qlora_adapter",
    "zip",
    ADAPTER_DIR,
)

# Reopen the completed training run to attach the final adapter artifacts.
with mlflow.start_run(run_id=MLFLOW_RUN_ID):
    mlflow.log_artifacts(ADAPTER_DIR, artifact_path="qlora_adapter")
    mlflow.log_artifact(ADAPTER_ARCHIVE)

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

print("Adapter directory:", ADAPTER_DIR)
print("Adapter archive:", ADAPTER_ARCHIVE)
print("Archive size (MB):", round(os.path.getsize(ADAPTER_ARCHIVE) / 1024**2, 2))
print("MLflow database:", MLFLOW_DB)
print("MLflow runs in experiment:", len(runs))
print("MLflow run ID:", MLFLOW_RUN_ID)


Adapter directory: /kaggle/working/cardioagent_qlora_adapter
Adapter archive: /kaggle/working/cardioagent_qlora_adapter.zip
Archive size (MB): 10.94
MLflow database: /kaggle/working/mlflow.db
MLflow runs in experiment: 1
MLflow run ID: c6cda821a7754d08bf9d604961f066e2


In [14]:

# CELL 10 - Export trained model to GGUF

GGUF_DIR = "/kaggle/working/cardioagent_qwen25_gguf"

try:
    if not hasattr(model, "save_pretrained_gguf"):
        raise AttributeError(
            "The loaded model does not expose save_pretrained_gguf."
        )

    model.save_pretrained_gguf(
        GGUF_DIR,
        tokenizer,
        quantization_method="q4_k_m"
    )

    print("GGUF export completed:", GGUF_DIR)
    print("Files:", os.listdir(GGUF_DIR))

except Exception as exc:
    print("GGUF conversion did not complete.")
    print("The previously saved LoRA adapter should remain available.")
    print(type(exc).__name__, ":", exc)


GGUF conversion did not complete.
The previously saved LoRA adapter should remain available.
RuntimeError : Unsloth: Not enough disk space to convert to GGUF.
The export needs about 30.4GB on the filesystem holding `/kaggle/working/cardioagent_qwen25_gguf`, which has 19.4GB free.
It writes a 16-bit merge, then a `f16` GGUF, then q4_k_m quants, and the merge and the intermediate are both still on disk while the quants are written.
Options: free space, export fewer quantization methods, point `save_directory` at a bigger filesystem, or push straight to Hugging Face with `.push_to_hub_gguf(...)`.
To skip this check set the environment variable UNSLOTH_DISK_PREFLIGHT=0.


In [22]:
GGUF_DIR = "/root/cardioagent_qwen25_gguf"

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m"
)

print("GGUF export completed!")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /root/cardioagent_qwen25_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (14 outputs left out)



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [01:09<00:00, 17.36s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:27<00:00,  6.95s/it]


Unsloth: Merge process complete. Saved to `/root/cardioagent_qwen25_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12757-mix-ef2df21 (app-b12757-mix-ef2df21-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/root/cardioagent_qwen25_gguf_gguf/Qwen2.5-7B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF con

In [24]:
import os
import shutil

SOURCE = "/root/cardioagent_qwen25_gguf_gguf"
DEST = "/kaggle/working/cardioagent_gguf"

os.makedirs(DEST, exist_ok=True)

for filename in [
    "Qwen2.5-7B-Instruct.Q4_K_M.gguf",
    "Modelfile"
]:
    shutil.copy2(
        os.path.join(SOURCE, filename),
        os.path.join(DEST, filename)
    )

print("Files prepared:")
for filename in os.listdir(DEST):
    path = os.path.join(DEST, filename)
    print(filename, round(os.path.getsize(path) / (1024**3), 2), "GB")

Files prepared:
Modelfile 0.0 GB
Qwen2.5-7B-Instruct.Q4_K_M.gguf 4.36 GB
